# Experimento: CNN sobre CIFAR-10 (PyTorch)

**Estudiante:** Rodrigo Armando Oviedo Vargas  
**Código de matrícula:** 022101068E → **semilla = 68** (dos últimos dígitos)

Este notebook contiene las pruebas, el análisis exploratorio (EDA) y la visualización de muestras.
El entrenamiento completo se ejecuta con `src/train.py`.

## 0. Configuración y semilla

In [ ]:
import sys
sys.path.append("..")   # para poder importar desde src/ (el notebook vive en notebooks/)

import numpy as np
import matplotlib.pyplot as plt
import torch
from torchvision import datasets

from src.dataset import (SEED, DATA_DIR, CLASSES, CIFAR_MEAN, CIFAR_STD,
                         set_seed, get_transforms, load_and_split, visualize_samples)
from src.model import CNN, count_parameters, test_forward_pass

set_seed(SEED)   # torch.manual_seed(68), np.random.seed(68), ...
print("Semilla aplicada:", SEED)
print("PyTorch:", torch.__version__, "| GPU disponible:", torch.cuda.is_available())

## 1. Carga del dataset CIFAR-10

In [ ]:
raw_train = datasets.CIFAR10(DATA_DIR, train=True, download=True)   # sin transformaciones (imágenes PIL)
raw_test = datasets.CIFAR10(DATA_DIR, train=False, download=True)

print("Imágenes de entrenamiento:", len(raw_train))
print("Imágenes de test         :", len(raw_test))
print("Clases                   :", raw_train.classes)
print("Tamaño de cada imagen    :", raw_train[0][0].size, "(ancho x alto) a color RGB")

## 2. EDA (análisis exploratorio)

In [ ]:
# 2.1 Distribución de clases (debe estar perfectamente balanceado: 5000 por clase)
counts = np.bincount(raw_train.targets)
plt.figure(figsize=(9, 3.5))
plt.bar(CLASSES, counts)
plt.title("Imágenes por clase en el set de entrenamiento")
plt.ylabel("Cantidad")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
print(dict(zip(CLASSES, counts.tolist())))

In [ ]:
# 2.2 Estadísticas por canal RGB (de aquí salen la media y desviación de Normalize)
data = raw_train.data / 255.0            # array (50000, 32, 32, 3) con valores en [0, 1]
print("Forma del array de imágenes:", data.shape, "-> (N, Alto, Ancho, Canales)")
print("Media por canal (R,G,B)     :", data.mean(axis=(0, 1, 2)).round(4), "| usada en Normalize:", CIFAR_MEAN)
print("Desv. estándar (R,G,B)      :", data.std(axis=(0, 1, 2)).round(4), "| usada en Normalize:", CIFAR_STD)
print("Valor mínimo / máximo       :", data.min(), "/", data.max())

## 3. Visualización de muestras con su etiqueta real

In [ ]:
visualize_samples(n=16, show=True)

## 4. Pipeline de transformaciones y Data Augmentation

In [ ]:
train_tf, eval_tf = get_transforms()
print("Transformaciones de ENTRENAMIENTO:\n", train_tf)
print("\nTransformaciones de VALIDACIÓN/TEST:\n", eval_tf)

In [ ]:
# Efecto del augmentation: la MISMA imagen original pasada 7 veces por RandomCrop + RandomHorizontalFlip
from torchvision import transforms
aug_only = transforms.Compose([transforms.RandomCrop(32, padding=4),
                               transforms.RandomHorizontalFlip(p=0.5)])
img, label = raw_train[0]
fig, axes = plt.subplots(1, 8, figsize=(16, 2.4))
axes[0].imshow(img); axes[0].set_title("Original"); axes[0].axis("off")
for ax in axes[1:]:
    ax.imshow(aug_only(img)); ax.set_title("Aumentada"); ax.axis("off")
plt.suptitle(f"Data Augmentation sobre una imagen de clase '{CLASSES[label]}'")
plt.tight_layout()
plt.show()

## 5. Partición estratificada train / validación

In [ ]:
train_set, val_set, test_set = load_and_split(val_size=0.1, seed=SEED)

img_t, lab_t = train_set[0]
print("\nUna muestra individual -> imagen:", tuple(img_t.shape), "(C, H, W) | etiqueta:", lab_t)

## 6. Arquitectura y validación de tensores 4D (forward pass)

In [ ]:
model = CNN()
print(model)
print("\nParámetros entrenables:", f"{count_parameters(model):,}")

In [ ]:
test_forward_pass(batch_size=8)   # imprime shapes intermedios y hace assert de la salida (8, 10)

## 7. Prueba rápida de aprendizaje (sanity check)

Si la red y el bucle de entrenamiento están bien, la pérdida debe **bajar** al entrenar repetidamente sobre un solo lote.

In [ ]:
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

set_seed(SEED)
loader = DataLoader(val_set, batch_size=64, shuffle=False)
images, labels = next(iter(loader))
print("Lote de prueba:", tuple(images.shape), tuple(labels.shape))

model = CNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

losses = []
for step in range(40):
    optimizer.zero_grad()
    loss = criterion(model(images), labels)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())

print(f"Loss inicial: {losses[0]:.4f}  ->  Loss final: {losses[-1]:.4f}")
plt.plot(losses); plt.title("Sanity check: sobreajuste a un solo lote")
plt.xlabel("Paso"); plt.ylabel("Loss"); plt.grid(alpha=0.3); plt.show()

## 8. Entrenamiento completo

El entrenamiento por épocas, las curvas de aprendizaje y la matriz de confusión se generan con:

```bash
python src/train.py
```

Los resultados quedan en `outputs/loss_accuracy.png` y `outputs/confusion_matrix.png`.